# MIL-CREDA frente a CREDA — fase uno: la corrida

Este cuaderno corre la campaña y nada más: el pronóstico de costo, la búsqueda del techo de cada familia y la campaña completa, en los dos niveles de contaminación que el informe muestra uno al lado del otro. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Report_v1.ipynb`, que lee `summary.json`, `runs.jsonl` y el registro de la búsqueda desde `MIL-CREDA/Results/Benchmark/` y nunca vuelve a entrenar nada.

Separar los dos es lo que hace barata una corrección al informe: re-renderizarlo cuesta segundos porque no toca la campaña, en vez del costo de cómputo entero.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json
import time

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a
    calcular acá: si esto y el archivo dijeran cosas distintas, habría dos
    versiones del mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe. `config.is_pilot_scale()` es la única lectura de esa regla --- las dos
# constantes que separan una escala de la otra son `EPOCHS` y `SEEDS` ---, y
# escribirla otra vez acá serían dos ortografías de lo mismo.
#
# Sin esto la reducción se construía sin `pilot`, o sea `False`: una corrida de
# tres épocas y una semilla escribía en `Results/Benchmark/`, que es el árbol de
# la corrida completa, y sus números quedaban ahí para que alguien los citara.
ES_ENSAYO = config.is_pilot_scale()
reduction = harness.Reduction(device=str(device), environment=harness.environment(),
                              pilot=ES_ENSAYO)

# Las dos pasadas de la campaña, declaradas donde se declara todo lo demás que
# decide dónde escribe esta corrida.
#
# Una campaña es cada transferencia a UNA tasa --- `results_for` lo dice de sí
# mismo, y el barrido de ruido es la forma opuesta, UNA transferencia a través de
# cada nivel ---, así que correr el nivel contaminado no es una campaña más
# grande ni un experimento nuevo: es una segunda pasada de esta misma forma. La
# campaña no lee nada del barrido y el barrido no gobierna nada de acá; hacer que
# dependiera de él favorecería a una de las dos estrategias, y por eso el barrido
# se mira y no se consulta.
#
# Los dos niveles salen de las dos constantes que el informe ya lee, y no de una
# lista derivada de `CHECKPOINT_LEVELS` ni de `NOISE_LEVELS`: `NOISE_REPORTED`
# declara de sí mismo que es «el nivel contaminado que el informe y el latente
# muestran al lado de 0.0», y esas dos son exactamente las pasadas que existen.
# Derivarlas de `CHECKPOINT_LEVELS` --- que hoy vale la misma pareja --- haría
# que un tercer nivel de pesos, que es una decisión del barrido, le agregara una
# pasada a la campaña sin que nadie la pidiera.
#
# Sin la segunda pasada, las celdas contaminadas del informe y la mitad
# contaminada del latente dicen «no hay corridas»: leen
# `results_for(NOISE_REPORTED, "campaign", ...)`, que ningún paso escribía.
NIVELES = (config.NOISE, config.NOISE_REPORTED)

shape = config.sizing()
print(json.dumps(shape, indent=2))
print()
print(harness.header(reduction))
print()
print("environment:", reduction.environment["platform"],
      "| torch", reduction.environment["torch"],
      "| self-hosted" if reduction.environment["selfHosted"] else "| hosted runtime")
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", " y ".join(
          str(config.results_for(nivel, reduction.kind, reduction.pilot))
          for nivel in NIVELES))

In [ ]:
# One run, timed, before committing to the whole grid. An estimate of the cost is
# cheaper than the cost, so it happens first.
from MIL_CREDA_Benchmark import bags

material = {role: bags.build(code, config.DATA_CACHE, config.SEEDS[0])
            for role, code in zip(("source", "target"), config.TRANSFERS[0])}
probe = harness.run_one("G", config.TRANSFERS[0], config.SEEDS[0],
                        reduction, device, material)
per_run = probe["seconds"]
# Por las dos pasadas y no por una: la rejilla corre entera una vez por nivel,
# así que lo que esta celda pronostica es `len(NIVELES)` veces la rejilla. Un
# pronóstico de una sola pasada diría la mitad de lo que el cuaderno va a gastar,
# que es peor que no pronosticar nada.
rejilla = shape["runs"] * len(NIVELES)
full = (len(config.ARMS) * len(config.TRANSFERS) * 30 * per_run * 20
        / reduction.epochs) * len(NIVELES)
# The search's own forecast, and it goes first because the search goes first. Its
# epochs are its own — never the pilot's — so the ratio is part of the estimate.
busqueda = shape["search"]
segundos = busqueda["runs"] * per_run * busqueda["epochs"] / reduction.epochs
print(f"one full arm, {reduction.epochs} epochs: {per_run:.1f}s")
print(f"the ceiling search ({busqueda['runs']} runs at {busqueda['epochs']} epochs): "
      f"about {segundos / 60:.0f} min"
      # Anuncia lo que hará la celda que obtiene los techos, dos más abajo, y
      # nada más que eso. En ensayo esa celda LEE el registro del ENSAYO y no
      # busca, así que acá no hay búsqueda que pronosticar. A escala completa sí
      # busca, y entonces la pregunta es por el registro COMPLETO: cada rama
      # pregunta por el archivo que su propia celda va a leer, que es la única
      # forma de que el pronóstico no hable de otra corrida.
      + ("  — en ensayo se lee el registro del ensayo, no se busca" if ES_ENSAYO
         else ("" if harness.search_record(pilot=False) is None
               else "  — already on record, skipped")))
print(f"this grid ({shape['runs']} runs x {len(NIVELES)} niveles = {rejilla} "
      f"runs): about {rejilla * per_run / 60:.0f} min")
print(f"at 20 epochs and 30 seeds: about {full / 3600:.0f} h")
del material, probe

## La búsqueda del techo

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación. Uno solo para las dos familias iguala el coeficiente y
desiguala el balance —los dos objetivos están separados por un factor `B_src`, así
que un mismo número pone la adaptación en la mayor parte de un objetivo y en una
décima parte del otro— por eso se busca **uno por familia**, y cada derivación
hereda el de la suya.

Es un experimento y se declara como tal. Corre sobre `SEARCH_TRANSFERS` y mide
sobre las bolsas de **validación**, nunca sobre el material del que se lee el
veredicto: elegir por resultado ahí haría que el veredicto informe una decisión que
él mismo ya tomó. La elección se hace por diferencias apareadas dentro de cada
`(semilla, transferencia)`, y un empate va al techo más chico —el mismo resultado
con menos adaptación es la afirmación más débil.

Y corre a **su** escala, no a la del piloto. La rampa sube sobre la fracción de
entrenamiento transcurrida: con tres épocas satura en la segunda y todo techo se
alcanza casi enseguida, así que un techo encontrado ahí describe un paisaje en el
que la campaña no entrena nunca. Es la única parte de este cuaderno que no tiene
escala de piloto, y por eso es la más larga.

Se busca una vez. Si `ceilings.json` ya existe se lee y no se vuelve a buscar: que
el registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es exactamente el refinanciamiento silencioso que la
campaña se niega a hacer. Para volver a empezar hay que borrarlo a mano.

In [ ]:
from dataclasses import replace

# Los techos vigentes. La `reduction` se reconstruye con lo que dice el registro
# y no con lo que quedó en memoria: `config.CEILINGS` se llena al importar, y si
# la búsqueda corre en este mismo proceso ese mapeo sigue vacío y la campaña se
# negaría con razón.
#
# En ensayo se LEEN y no se buscan. La búsqueda de ensayo es un paso propio
# (`search-pilot`), y su archivo, `ceilings.pilot.json`, es raíz declarada de ese
# paso: una campaña de ensayo que además buscara escribiría en el árbol del paso
# de al lado, que es la falla que la declaración de `produces` existe para hacer
# visible.
#
# Y se leen los de ESTA escala, nombrada. Las dos ramas leen ahora el registro de
# `reduction.pilot` y por eso dan lo mismo; antes daban lo mismo por otra razón
# --- las dos leían el VIGENTE ---, y esa razón era el defecto: una campaña de
# ensayo corría bajo los techos de la búsqueda completa, medidos a veinte épocas
# en otro experimento, mientras `ceilings.pilot.json` ---escrito por el paso de
# al lado, dos posiciones antes en este mismo recorrido--- quedaba en disco sin
# que nada lo leyera. La escala es un modo del recorrido entero: en ensayo todo
# corre y consume lo del ensayo.
#
# A escala completa se busca, como siempre: este es el único lugar del recorrido
# donde `ceilings.json` puede nacer, y sacar la búsqueda de acá dejaría a la
# campaña completa sin techos y sin quien se los busque.
#
# Y acá, UNA sola vez, arriba del bucle de niveles: los techos se buscaron en
# limpio y se mantienen fijos en las dos pasadas, así que la contaminada corre
# bajo el coeficiente elegido sin contaminación. Resolverlos adentro del bucle
# daría los mismos números hoy y, a escala completa, volvería a preguntar por el
# registro una vez por nivel: la puerta por la que la búsqueda entera --- nueve
# horas y media que nadie autorizó --- entra sin decir que está entrando. Lo que
# la contaminación le cuesta al techo lo separa `noise-diagnostic`, que re-busca
# a su nivel y no toca este registro.
if ES_ENSAYO:
    reduction = replace(
        reduction,
        ceilings=config.ceilings_on_record(pilot=reduction.pilot),
        ceilingsByTransfer=config.ceilings_by_transfer_on_record(
            pilot=reduction.pilot))
else:
    reduction = harness.with_ceilings_in_force(reduction, device)

## La corrida

Una pasada por nivel: la limpia y la contaminada, la misma rejilla entera las dos veces. Los techos ya están elegidos arriba y no se vuelven a tocar, así que lo único que cambia entre las dos es la tasa que lleva el material de entrenamiento.

Una línea por transferencia, no una por corrida: con treinta semillas la lista
completa serían mil ochocientas líneas y todo lo que dicen ya está en las tablas
de más abajo. Esta salida existe para saber que la campaña sigue viva, nada más.

In [ ]:
seen = {"n": 0}

def progress(line: str) -> None:
    seen["n"] += 1
    if seen["n"] % len(config.ARMS) == 0:
        print(f"  {seen['n']:>5}/{shape['runs']} corridas "
              f"({(time.perf_counter() - started) / 60:.1f} min)")

# Una pasada por nivel, y las dos son LA campaña. `replace` sobre la reducción de
# arriba y no una `Reduction` nueva: los techos ya están adentro de ella,
# resueltos una sola vez antes del bucle, y armar otra acá los volvería a pedir
# --- que a escala completa es la búsqueda entera, lanzada sin autorización.
#
# El contador de progreso se reinicia por pasada porque `shape["runs"]` es el
# tamaño de UNA rejilla: dejarlo correr diría `1200/600`.
resumenes = {}
for nivel in NIVELES:
    pasada = replace(reduction, labelNoise=nivel)
    # De la reducción con la que se está por correr, no de la constante: las tres
    # coordenadas de la primera pasada coinciden con `config.NOISE` por
    # casualidad, y esa casualidad es cómo se lee el árbol equivocado.
    raiz = config.results_for(pasada.labelNoise, pasada.kind, pasada.pilot)
    print(f"\ncampaña a ρ={nivel:g} → {raiz}")
    seen["n"] = 0
    started = time.perf_counter()
    summary = harness.campaign(pasada, device, progress=progress)
    runs = [json.loads(line) for line in
            (raiz / "runs.jsonl").read_text().splitlines() if line.strip()]
    print(f"campaña a ρ={nivel:g} terminada en "
          f"{(time.perf_counter() - started) / 60:.1f} min, {len(runs)} corridas")
    resumenes[f"{nivel:g}"] = summary